# Train the rocket controller offline

Train the current `example_controller.py` on `landings.csv`, evaluate action predictions on held-out flights, and save the fitted model for the simulator. The training data is resampled to give rarer actions more examples.

Restart the kernel and rerun all cells after updating the controller or physics. Saved output below may be from an earlier model. Retrain and save new weights before using this version. Use training data generated with the current physics and separate from `validation_runs.csv`.


In [19]:
from pathlib import Path
import sys
import numpy as np

# Works when launched from the exercise folder or the repository root.
exercise_dir = Path.cwd()
if not (exercise_dir / "example_controller.py").exists():
    exercise_dir = exercise_dir / "classification_exercise_temp"
exercise_dir = exercise_dir.resolve()
if not (exercise_dir / "example_controller.py").is_file():
    raise FileNotFoundError("Open this notebook from the exercise folder or repository root.")
sys.path.insert(0, str(exercise_dir))
from example_controller import Controller


## Settings and data

The validation split is by `run_id`, so observations from the same flight cannot occur in both sets. Keep the rows in their original order: the controller uses consecutive observations to calculate history features. Both successful and unsuccessful flights are retained, matching the simulator's training path.


In [20]:
data_path = exercise_dir / "landings.csv"
weights_path = exercise_dir / "weights.npz"
learning_rate = 0.1
epochs = 1000
validation_fraction = 0.2
max_copies = 3  # Maximum expected total copies per training observation; 1 disables resampling.
seed = 42

observation_columns = (
    "x", "y", "vx", "vy", "theta", "omega", "fuel_mass",
    "actual_thrust", "actual_gimbal", "wind_x", "wind_y", "air_density",
)
data = np.atleast_1d(np.genfromtxt(data_path, delimiter=",", names=True, dtype=None, encoding="utf-8"))
required = {*observation_columns, "action", "run_id", "step"}
missing = required - set(data.dtype.names or ())
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if not len(data):
    raise ValueError("The training CSV is empty.")
X = np.column_stack([data[name] for name in observation_columns]).astype(float)
y = np.asarray(data["action"])
run_ids = np.asarray(data["run_id"])
steps = np.asarray(data["step"])

unique_runs = np.unique(run_ids)
if len(unique_runs) < 2:
    raise ValueError("At least two distinct flights are needed for a validation split.")
if not 0 < validation_fraction < 1:
    raise ValueError("validation_fraction must be between 0 and 1.")
rng = np.random.default_rng(seed)
shuffled_runs = rng.permutation(unique_runs)
validation_count = min(len(unique_runs) - 1, max(1, round(len(unique_runs) * validation_fraction)))
validation_mask = np.isin(run_ids, shuffled_runs[:validation_count])
train_mask = ~validation_mask
print(f"Training: {train_mask.sum()} observations, {len(unique_runs) - validation_count} flights")
print(f"Validation: {validation_mask.sum()} observations, {validation_count} flights")
classes, counts = np.unique(y[train_mask], return_counts=True)
print("Training action counts:", dict(zip(classes.tolist(), counts.tolist())))


Training: 103680 observations, 800 flights
Validation: 26328 observations, 200 flights
Training action counts: {0: 14144, 1: 2455, 2: 68394, 3: 2523, 4: 1527, 5: 10900, 6: 1659, 7: 735, 8: 636, 9: 707}


## Resample the training flights

Rare actions are sampled more often using the square root of the inverse class frequency, capped by `max_copies`. Each extra target includes its preceding observation when available, preserving the history feature for the target. These short copies have their own run IDs, so history never crosses between them. Their first observation has no earlier history; the added context rows can also change class counts, so the result is deliberately approximate rather than perfectly balanced.

The validation set keeps its original distribution. Set `max_copies = 1` above and rerun to compare with the unresampled baseline.


In [21]:
if max_copies < 1:
    raise ValueError("max_copies must be at least 1.")
X_train, y_train = X[train_mask], y[train_mask]
original_ids, train_steps = run_ids[train_mask], steps[train_mask]
# Integer IDs also work if the CSV uses string run IDs.
_, train_ids = np.unique(original_ids, return_inverse=True)
classes, counts = np.unique(y_train, return_counts=True)
copy_rates = np.minimum(max_copies, np.sqrt(counts.max() / counts))
rate_by_action = dict(zip(classes, copy_rates))

# Stochastically round the desired extra copies, using the fixed seed above.
extra_rates = np.array([rate_by_action[action] - 1 for action in y_train])
extra_copies = np.floor(extra_rates).astype(int)
extra_copies += rng.random(len(y_train)) < (extra_rates - extra_copies)
indices, new_ids = [], []
next_id = len(unique_runs)
for i in np.flatnonzero(extra_copies):
    has_previous = (i > 0 and original_ids[i] == original_ids[i - 1]
                    and train_steps[i] == train_steps[i - 1] + 1)
    for _ in range(extra_copies[i]):
        if has_previous:
            indices.append(i - 1)
            new_ids.append(next_id)
        indices.append(i)
        new_ids.append(next_id)
        next_id += 1

indices = np.asarray(indices, dtype=int)
X_fit = np.concatenate((X_train, X_train[indices]))
y_fit = np.concatenate((y_train, y_train[indices]))
fit_ids = np.concatenate((train_ids, np.asarray(new_ids, dtype=int)))
fit_steps = np.concatenate((train_steps, train_steps[indices]))
resampled_classes, resampled_counts = np.unique(y_fit, return_counts=True)
print(f"Training observations: {len(y_train)} → {len(y_fit)}")
print("Resampled action counts:", dict(zip(resampled_classes.tolist(), resampled_counts.tolist())))


Training observations: 103680 → 210527
Resampled action counts: {0: 48644, 1: 9269, 2: 91492, 3: 9459, 4: 5721, 5: 30329, 6: 6144, 7: 3351, 8: 2914, 9: 3204}


## Fit the controller

Change the settings above to experiment. `fit` receives run IDs and step numbers so it does not construct history across flight boundaries, copied snippets, or missing steps.


In [22]:
controller = Controller(learning_rate=learning_rate, epochs=epochs)
controller.fit(X_fit, y_fit, run_ids=fit_ids, steps=fit_steps)
print(f"\nTraining loss: {controller.loss_history[0]:.4f} → {controller.loss_history[-1]:.4f}")


Epoch 1000/1000
Training loss: 2.3026 → 0.9132


## Validate on held-out flights

Use `predict_action`, as the simulator does, and reset its history at flight boundaries and gaps. Macro F1 treats each action present in the validation targets or predictions equally. These action metrics do not measure whether a landing succeeds.


In [23]:
X_val, y_val = X[validation_mask], y[validation_mask]
val_ids, val_steps = run_ids[validation_mask], steps[validation_mask]
controller.reset()
predictions = []
for i, observation in enumerate(X_val):
    if i and (val_ids[i] != val_ids[i - 1] or val_steps[i] != val_steps[i - 1] + 1):
        controller.reset()
    predictions.append(controller.predict_action(observation))
predictions = np.asarray(predictions)
labels = np.union1d(y_val, predictions)
f1 = []
for label in labels:
    tp = np.sum((predictions == label) & (y_val == label))
    fp = np.sum((predictions == label) & (y_val != label))
    fn = np.sum((predictions != label) & (y_val == label))
    f1.append(2 * tp / (2 * tp + fp + fn))
print(f"Validation accuracy: {np.mean(predictions == y_val):.3f}")
print(f"Validation macro F1: {np.mean(f1):.3f}")
print("Per-action F1:", {int(label): round(float(score), 3) for label, score in zip(labels, f1)})
controller.reset()


Validation accuracy: 0.778
Validation macro F1: 0.356
Per-action F1: {0: 0.861, 1: 0.302, 2: 0.879, 3: 0.341, 4: 0.056, 5: 0.103, 6: 0.101, 7: 0.385, 8: 0.185, 9: 0.347}


## Save for the simulator

This saves the model trained on the training flights above (the validation flights are not added back). In **Run a controller**, upload `example_controller.py` and the saved `weights.npz`, then select **Evaluate saved model**. Running this cell again overwrites the file.


In [25]:
controller.save(weights_path)
print(f"Saved model to {weights_path}")


Saved model to C:\Users\cararc\Documents\Personal\Archie-C.github.io\classification_exercise_temp\weights.npz
